# 00 — Preparazione dello split comune (dataset RF reale)

Va eseguito **una volta**. Produce `splits/split_files_v2.npz`, che tutti i notebook dei modelli caricano con `qdetect.load_split`: ogni modello (RF, MLP, VQE, QAOA, VQS, stump, QUBO, ...) vede le stesse righe e lo stesso preprocessing.

Il dataset arriva già diviso in `train_dataset.csv` e `test_dataset.csv`. Quasi tutte le righe del test, però, hanno un vettore di feature **identico** a una riga del train: valutare sul test così com'è misura soprattutto la memorizzazione. Per questo:

- **train / validation** vengono dal file di training, divisi per gruppi di vettori identici;
- **test** = righe del file di test il cui vettore **non compare mai** nel file di training (test "nuovo");
- **test_original** = campione del test così com'è, conservato solo per confronto con i risultati già ottenuti.

In [1]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import qdetect as qd
print('repo:', REPO)

repo: /Users/leonardo/Desktop/Quantum-Enhanced-Detection-Mechanisms-for-Mobile-Network-Security


In [2]:
# Cartella con train_dataset.csv e test_dataset.csv (sul server: output/split).
DATA_DIR = Path(os.environ.get('QDETECT_DATA_DIR', 'output/split')).expanduser()
TRAIN_CSV = DATA_DIR / 'train_dataset.csv'
TEST_CSV = DATA_DIR / 'test_dataset.csv'
OUT = REPO / 'splits' / 'split_files_v2.npz'

CAP_TRAIN = 120_000      # righe per classe dal file di training
CAP_TEST = 60_000        # righe per classe dal test "nuovo"
CAP_TEST_ORIG = 10_000   # righe per classe dal test originale (solo confronto)
SAMPLE_SEED, SPLIT_SEED = 2026, 42
SIZES = dict(n_train=5_000, n_train_large=20_000, n_val=1_000, n_test=5_000)

for p in (TRAIN_CSV, TEST_CSV):
    assert p.exists(), f'Non trovato: {p.resolve()}'
    print(p.resolve(), round(p.stat().st_size / 1e9, 2), 'GB')

/Users/leonardo/Downloads/train_dataset.csv 2.05 GB
/Users/leonardo/Downloads/test_dataset.csv 0.51 GB


## 1. Quanto il test si sovrappone al train

`lookup_accuracy_on_seen`: accuratezza di una tabella che, per ogni vettore del test già presente nel train, risponde con l'etichetta più frequente nel train. Nessun modello: solo memoria.

In [3]:
t0 = time.time()
overlap = qd.overlap_report(TRAIN_CSV, TEST_CSV)
print(f'{time.time()-t0:.0f} s')
print(json.dumps(overlap, indent=1))

24 s
{
 "train_rows": 52752000,
 "train_unique_vectors": 3289761,
 "test_rows": 13188000,
 "test_unique_vectors": 1714416,
 "test_unique_vectors_in_train": 1375424,
 "test_rows_with_vector_in_train": 12823655,
 "test_rows_frac_in_train": 0.9723729905975129,
 "test_rows_novel": 364345,
 "lookup_accuracy_on_seen": 0.9925100137207372
}


## 2. Campionamento

In [4]:
t0 = time.time()
train_hashes = qd.vector_hashes(TRAIN_CSV)
train_df, s_train = qd.sample_per_class(TRAIN_CSV, CAP_TRAIN, seed=SAMPLE_SEED)
test_df, s_test = qd.sample_per_class(TEST_CSV, CAP_TEST, seed=SAMPLE_SEED + 1, exclude_groups=train_hashes)
orig_df, s_orig = qd.sample_per_class(TEST_CSV, CAP_TEST_ORIG, seed=SAMPLE_SEED + 2)
print(f'{time.time()-t0:.0f} s')
print('vettori unici nel train:', len(train_hashes))
for name, st in [('train', s_train), ('test nuovo', s_test), ('test originale', s_orig)]:
    print(name, json.dumps(st))

30 s
vettori unici nel train: 3289761
train {"rows_in_file": 52752000, "rows_per_class_in_file": {"0": 26376000, "1": 26376000}, "rows_dropped_bad_label": 0, "rows_excluded_by_group": 0, "sample_seed": 2026, "cap_per_class": 120000}
test nuovo {"rows_in_file": 13188000, "rows_per_class_in_file": {"0": 212420, "1": 151925}, "rows_dropped_bad_label": 0, "rows_excluded_by_group": 12823655, "sample_seed": 2027, "cap_per_class": 60000}
test originale {"rows_in_file": 13188000, "rows_per_class_in_file": {"0": 6594000, "1": 6594000}, "rows_dropped_bad_label": 0, "rows_excluded_by_group": 0, "sample_seed": 2028, "cap_per_class": 10000}


## 3. Diagnostica sul campione di training

`bayes_error_floor`: errore minimo inevitabile per qualunque modello che usi queste 8 feature, dovuto ai vettori identici con etichette diverse.

In [5]:
dup = qd.duplicate_report(train_df[qd.FEATURES].to_numpy(float), train_df[qd.TARGET].to_numpy())
print(json.dumps(dup, indent=1))
display(pd.crosstab(train_df['freq1'], train_df[qd.TARGET], normalize='index').round(3))

{
 "rows": 240000,
 "unique_vectors": 144931,
 "duplicate_row_fraction": 0.39612083333333337,
 "conflicting_vectors": 264,
 "rows_in_conflicting_vectors": 738,
 "bayes_error_floor": 0.0011458333333333333
}


label_encoded,0,1
freq1,,
5180,0.498,0.502
5200,0.475,0.525
5240,0.493,0.507
5745,0.480,0.520
5765,0.474,0.526
5785,0.497,0.503
5805,0.502,0.498


In [6]:
# Il file è in ordine di acquisizione? Frazione di malevoli per blocchi del file.
# Se è costante (≈0,5 ovunque) il file è stato rimescolato: l'ordine delle righe
# non contiene informazione temporale e uno split 'a blocchi' non avrebbe senso.
blocks = pd.cut(train_df['row_id'], 20, labels=False)
print(train_df.groupby(blocks)[qd.TARGET].mean().round(3).tolist())

[0.501, 0.497, 0.5, 0.497, 0.498, 0.503, 0.505, 0.507, 0.509, 0.501, 0.494, 0.494, 0.504, 0.496, 0.499, 0.497, 0.5, 0.495, 0.505, 0.498]


## 4. Split e salvataggio

In [7]:
split = qd.make_split_from_files(train_df, test_df, orig_df, seed=SPLIT_SEED, **SIZES)
qd.save_split(split, OUT, extra_meta={'train_csv': str(TRAIN_CSV), 'test_csv': str(TEST_CSV),
                                      'overlap': overlap, 'duplicates': dup,
                                      'sampling': {'train': s_train, 'test': s_test, 'test_original': s_orig},
                                      'sizes': SIZES})
print('salvato:', OUT, '| pool:', split.meta['pool_sizes'])
split.summary()

salvato: /Users/leonardo/Desktop/Quantum-Enhanced-Detection-Mechanisms-for-Mobile-Network-Security/splits/split_files_v2.npz | pool: {'train': 215949, 'val': 24051, 'test': 120000}


,set,n,frac_malicious
0,train,5000,0.49940
1,train_large,20000,0.49945
2,val,1000,0.50500
3,test,5000,0.50000
4,test_original,5000,0.50000


## 5. Controllo: test originale contro test nuovo

Stessi modelli, addestrati sulle 20.000 righe di `train_large`, valutati su entrambi i test. La differenza misura quanto il test originale premia la memorizzazione. Non sono risultati della tesi, solo un controllo.

In [8]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

P = qd.prepare(split, fit_on='train_large')
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0),
    'Gradient Boosting': HistGradientBoostingClassifier(random_state=0),
}
rows = []
for name, m in models.items():
    m.fit(P.X_train_large, P.y_train_large)
    for test_name, (X, y) in [('test nuovo', (P.X_test, P.y_test)), ('test originale', P.extra['test_original'])]:
        r = qd.evaluate(name, y, m.predict_proba(X)[:, 1], family='classical', split=test_name,
                        n_train=len(P.y_train_large))
        rows.append(r)
check = pd.DataFrame(rows)[['model', 'split', 'auroc', 'auroc_lo', 'auroc_hi', 'accuracy', 'f1', 'pd_at_pfa_0.01']]
check.round(4)

,model,split,auroc,auroc_lo,auroc_hi,accuracy,f1,pd_at_pfa_0.01
0,Logistic Regression,test nuovo,0.9019,0.8932,0.9105,0.8326,0.8405,0.1412
1,Logistic Regression,test originale,0.9938,0.9923,0.9954,0.9674,0.9675,0.8660
2,Random Forest,test nuovo,0.9760,0.9727,0.9793,0.9140,0.9163,0.6528
3,Random Forest,test originale,0.9991,0.9987,0.9995,0.9860,0.9860,0.9800
4,Gradient Boosting,test nuovo,0.9755,0.9718,0.9792,0.9268,0.9280,0.4736
5,Gradient Boosting,test originale,0.9992,0.9989,0.9996,0.9882,0.9882,0.9868


## Uso nei notebook dei modelli

```python
import qdetect as qd
split = qd.load_split(REPO / 'splits/split_files_v2.npz')
P = qd.prepare(split, angle_scale=2.0)              # 8 feature → angoli per i circuiti
# P = qd.prepare(split, k_best=4, angle_scale=2.0)  # variante a 4 qubit
...
row = qd.evaluate('QAOA 8q', P.y_test, scores, family='quantum', seed=seed,
                  n_train=len(P.y_train), n_params=n_params,
                  train_seconds=t_train, infer_seconds=t_inf)
qd.append_results(REPO / 'results/all_results.csv', [row])
```

Il test di riferimento è `P.X_test` (vettori mai visti nel training). `P.extra['test_original']` serve solo a mostrare quanto cambiano i numeri sul test originale. Il punteggio passato a `evaluate` è **alto = malevolo**; per i modelli a energia il segno si fissa con `qd.orient_scores` su train o validation.